In [1]:
import ssl

ssl._create_default_https_context = ssl._create_stdlib_context

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import os
from PIL import Image

TRAIN_DIR = 'data/train/images' 
TEST_DIR = 'data/test/images'
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class FilthDataset(Dataset):
    def __init__(self, root_dir, clean_transform=None, filth_transform=None):
        self.root_dir = root_dir
        self.clean_transform = clean_transform
        self.filth_transform = filth_transform
        self.image_paths = []
        self.labels = []
        
        valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')

        for root, _, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(valid_extensions):
                    self.image_paths.append(os.path.join(root, file))
                    if 'clean' in file.lower():
                        self.labels.append(0) # Clean
                    else:
                        self.labels.append(1) # Filthy

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        # Apply heavy transforms to filth, standard transforms to clean
        if label == 1 and self.filth_transform:
            image = self.filth_transform(image)
        elif self.clean_transform:
            image = self.clean_transform(image)
            
        return image, label

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

clean_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.1, 0.5)), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    normalize
])
filth_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.1, 0.5)), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=15),
    transforms.ToTensor(),
    normalize
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize
])

print("\n--- Loading Custom Datasets ---")
full_train_dataset = FilthDataset(TRAIN_DIR, clean_transform=clean_transforms, filth_transform=filth_transforms)

full_val_dataset = FilthDataset(TRAIN_DIR, clean_transform=test_transforms, filth_transform=test_transforms)

num_train = len(full_train_dataset)
indices = list(range(num_train))
np.random.seed(123)
np.random.shuffle(indices)

split = int(np.floor(0.15 * num_train))
train_idx, val_idx = indices[split:], indices[:split]

train_data = Subset(full_train_dataset, train_idx)
val_data = Subset(full_val_dataset, val_idx)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

test_dataset = FilthDataset(TEST_DIR, clean_transform=test_transforms, filth_transform=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

class_counts = [0, 0]
for idx in train_idx:
    label = full_train_dataset.labels[idx]
    class_counts[label] += 1

num_clean, num_filthy = class_counts[0], class_counts[1]
pos_weight_val = num_clean / max(1, num_filthy)
pos_weight = torch.tensor([pos_weight_val]).to(device)


print("\n--- Building ResNet50 Model ---")
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(num_ftrs, 1)
)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

print("\n--- Starting Training ---")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        
    epoch_train_loss = running_loss / len(train_data)
    
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            
    epoch_val_loss = val_loss / len(val_data)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

torch.save(model.state_dict(), 'resnet50_filth_classifier.pth')

print("\n--- Evaluation ---")
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        probs = torch.sigmoid(model(inputs))
        preds = (probs > 0.5).int().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=['Clean (0)', 'Filthy (1)']))
cm = confusion_matrix(all_labels, all_preds)
print(f"True Negatives : {cm[0][0]}\nFalse Positives: {cm[0][1]}\nFalse Negatives: {cm[1][0]}\nTrue Positives : {cm[1][1]}")

Using device: cuda

--- Loading Custom Datasets ---

--- Building ResNet50 Model ---


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/ugnius/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:18<00:00, 5.68MB/s]


--- Starting Training ---


Epoch 01/20 | Train Loss: 0.4708 | Val Loss: 0.4642
Epoch 02/20 | Train Loss: 0.4446 | Val Loss: 0.4536
Epoch 03/20 | Train Loss: 0.4212 | Val Loss: 0.4471
Epoch 04/20 | Train Loss: 0.3919 | Val Loss: 0.4405
Epoch 05/20 | Train Loss: 0.3781 | Val Loss: 0.4339
Epoch 06/20 | Train Loss: 0.3596 | Val Loss: 0.4276
Epoch 07/20 | Train Loss: 0.3421 | Val Loss: 0.4247
Epoch 08/20 | Train Loss: 0.3305 | Val Loss: 0.4238
Epoch 09/20 | Train Loss: 0.3095 | Val Loss: 0.4200
Epoch 10/20 | Train Loss: 0.2991 | Val Loss: 0.4149
Epoch 11/20 | Train Loss: 0.2794 | Val Loss: 0.4131
Epoch 12/20 | Train Loss: 0.2722 | Val Loss: 0.4141
Epoch 13/20 | Train Loss: 0.2642 | Val Loss: 0.4069
Epoch 14/20 | Train Loss: 0.2514 | Val Loss: 0.4062
Epoch 15/20 | Train Loss: 0.2423 | Val Loss: 0.4088
Epoch 16/20 | Train Loss: 0.2349 | Val Loss: 0.4126
Epoch 17/20 | Train Loss: 0.2240 | Val Loss: 0.4071
Epoch 18/20 | Train Loss: 0.2216 | Val Loss: 0.4035
Epoch 19/20 | Train Loss: 0.2188 | Val Loss: 0.4105
Epoch 20/20 

In [ ]:
import cv2
import numpy as np
import os
import glob
import torch
import torch.nn as nn
from torchvision import models, transforms
from pytorch_grad_cam import GradCAMPlusPlus
import matplotlib.pyplot as plt

TEST_IMAGES_DIR = 'data/test/images'
TEST_MASKS_DIR = 'data/test/masks'
OUTPUT_VIS_DIR = 'output_visualizations'
MODEL_WEIGHTS = 'mobilenet_filth_classifier_pytorch.pth'

T_OFFSET = 50
RADIUS = 20
EPS_VAL = 10

IMG_SIZE = 224
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_rgb888_mask_as_binary(mask_path, target_size=(224, 224)):
    mask = cv2.imread(mask_path, cv2.IMREAD_COLOR)
    mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
    gray_mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    return (gray_mask > 0).astype(np.uint8)

def calculate_iou(pred_mask, true_mask):
    intersection = np.logical_and(pred_mask, true_mask).sum()
    union = np.logical_or(pred_mask, true_mask).sum()
    return intersection / union if union != 0 else 1.0

def apply_guided_filter(rgb_image, rough_mask, radius, eps):
    mask_float = rough_mask.astype(np.float32)
    guide_float = rgb_image.astype(np.float32) / 255.0
    refined = cv2.ximgproc.guidedFilter(guide_float, mask_float, radius, eps)
    return (refined > 0.5).astype(np.uint8)

def refine_mask(heatmap, rgb_image, t_offset_val, radius, eps_val):
    max_h, mean_h = np.max(heatmap), np.mean(heatmap)
    base_t = 1.0 - (max_h - mean_h)
    t_offset = (t_offset_val - 50) / 100.0
    final_t = np.clip(base_t + t_offset, 0.01, 0.99)
    binary_mask = (heatmap >= final_t).astype(np.uint8)
    true_eps = eps_val * 1e-4 
    return apply_guided_filter(rgb_image, binary_mask, radius, true_eps)


def run_batch(img_paths, mask_paths, model, cam, transform):
    os.makedirs(OUTPUT_VIS_DIR, exist_ok=True)
    total_iou = 0.0
    
    print(f"Starting batch process: {len(img_paths)} images...")
    
    for i, (img_path, mask_path) in enumerate(zip(img_paths, mask_paths)):
        bgr_img = cv2.imread(img_path)
        bgr_img = cv2.resize(bgr_img, (IMG_SIZE, IMG_SIZE))
        rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
        gt_mask = load_rgb888_mask_as_binary(mask_path, (IMG_SIZE, IMG_SIZE))
        
        input_tensor = transform(rgb_img).unsqueeze(0).to(device)
        heatmap = cam(input_tensor=input_tensor, targets=None)[0, :]
        
        pred_mask = refine_mask(heatmap, rgb_img, T_OFFSET, RADIUS, EPS_VAL)
        iou = calculate_iou(pred_mask, gt_mask)
        total_iou += iou

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        overlay_pred = bgr_img.copy()
        overlay_pred[pred_mask == 1] = [0, 0, 255] 
        axes[0].imshow(cv2.cvtColor(overlay_pred, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Prediction (Red)\nIoU: {iou:.4f}")
        
        overlay_gt = bgr_img.copy()
        overlay_gt[gt_mask == 1] = [0, 255, 0]
        axes[1].imshow(cv2.cvtColor(overlay_gt, cv2.COLOR_BGR2RGB))
        axes[1].set_title("Ground Truth (Green)")
        
        axes[2].imshow(heatmap, cmap='jet')
        axes[2].set_title("Grad-CAM++ Heatmap")
        
        for ax in axes: ax.axis('off')
        
        base_name = os.path.basename(img_path)
        plt.savefig(os.path.join(OUTPUT_VIS_DIR, f"res_{base_name}"))
        plt.close()

        if (i+1) % 10 == 0:
            print(f"Processed {i+1}/{len(img_paths)}... Mean IoU: {total_iou/(i+1):.4f}")

    print(f"\nDONE. Final Mean IoU: {total_iou/len(img_paths):.4f}")
    print(f"Visualizations saved in: {OUTPUT_VIS_DIR}")


img_paths = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, '*.[jJ][pP]*[gG]')))
mask_paths = sorted(glob.glob(os.path.join(TEST_MASKS_DIR, '*.[pP][nN][gG]')))

if not img_paths:
    print(f"Error: No images found in {TEST_IMAGES_DIR}")
elif len(img_paths) != len(mask_paths):
    print(f"Error: Mismatch! Found {len(img_paths)} images and {len(mask_paths)} masks.")
else:
    model = models.mobilenet_v2(pretrained=False)
    model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(model.last_channel, 1))
    model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=device))
    model = model.to(device).eval()

    cam = GradCAMPlusPlus(model=model, target_layers=[model.features[-1]])
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    run_batch(img_paths, mask_paths, model, cam, transform)

/home/ugnius/Documents/KTU/IV-kursas/II-semestras/BBP/V2/model_tests/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ugnius/Documents/KTU/IV-kursas/II-semestras/BBP/V2/model_tests/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Starting batch process: 75 images...
Processed 10/75... Mean IoU: 0.0000
Processed 20/75... Mean IoU: 0.0000
Processed 30/75... Mean IoU: 0.0339
Processed 40/75... Mean IoU: 0.0813
Processed 50/75... Mean IoU: 0.1188
Processed 60/75... Mean IoU: 0.1385
Processed 70/75... Mean IoU: 0.1483

DONE. Final Mean IoU: 0.1534
Visualizations saved in: output_visualizations


In [1]:
import cv2
import numpy as np
import os
import glob
import torch
import torch.nn as nn
from torchvision import models, transforms
from pytorch_grad_cam import ScoreCAM

TEST_IMAGES_DIR = 'data/test/images'
MODEL_WEIGHTS = 'mobilenet_filth_classifier_pytorch.pth'
OUTPUT_DIR = 'scorecam_results'

IMG_SIZE = 224
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUTPUT_DIR, exist_ok=True)

T_OFFSET = 50
RADIUS = 5
EPS_VAL = 3


print("Loading PyTorch Model...")
model = models.mobilenet_v2(pretrained=False)
model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(model.last_channel, 1))
model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=device))
model = model.to(device).eval()

cam = ScoreCAM(model=model, target_layers=[model.features[-1]])

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


img_paths = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, '*.[jJ][pP]*[gG]')))
if not img_paths:
    raise ValueError(f"No images found in {TEST_IMAGES_DIR}")

print(f"\nProcessing {len(img_paths)} images with Score-CAM...")

for i, img_path in enumerate(img_paths):
    filename = os.path.basename(img_path)
    
    orig_bgr = cv2.imread(img_path)
    h, w = orig_bgr.shape[:2]
    
    model_input_rgb = cv2.cvtColor(cv2.resize(orig_bgr, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
    input_tensor = transform(model_input_rgb).unsqueeze(0).to(device)
    
    heatmap_224 = cam(input_tensor=input_tensor, targets=None)[0, :]
    
    heatmap_500 = cv2.resize(heatmap_224, (w, h))
    
    max_h, mean_h = np.max(heatmap_500), np.mean(heatmap_500)
    base_t = 1.0 - (max_h - mean_h)
    final_t = np.clip(base_t + (T_OFFSET - 50) / 100.0, 0.05, 0.95)
    binary_mask = (heatmap_500 >= final_t).astype(np.float32)
    
    guide_float = orig_bgr.astype(np.float32) / 255.0
    true_eps = EPS_VAL * 1e-4 
    refined = cv2.ximgproc.guidedFilter(guide_float, binary_mask, RADIUS, true_eps)
    final_discrete_mask = (refined > 0.5).astype(np.uint8)
    
    res = orig_bgr.copy()
    res[final_discrete_mask == 1] = [0, 0, 255] # Red overlay for filth
    
    cv2.imwrite(os.path.join(OUTPUT_DIR, f"scorecam_{filename}"), res)
    
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/{len(img_paths)}...")

print(f"\nDone! Results saved in '{OUTPUT_DIR}'.")

Loading PyTorch Model...


/home/ugnius/Documents/KTU/IV-kursas/II-semestras/BBP/V2/model_tests/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ugnius/Documents/KTU/IV-kursas/II-semestras/BBP/V2/model_tests/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



Processing 75 images with Score-CAM...


100%|██████████| 80/80 [00:00<00:00, 111.15it/s]


Processed 10/75...


...

Processed 60/75...


100%|██████████| 80/80 [00:00<00:00, 110.93it/s]


Processed 70/75...


100%|██████████| 80/80 [00:00<00:00, 111.06it/s]


Done! Results saved in 'scorecam_results'.
